<a href="https://colab.research.google.com/github/francji1/01ZLMA/blob/main/code/01ZLMA_ex07_Binary_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01ZLMA - Exercise 07

Exercise 07 of the course [01ZLMA](https://math.fel.cvut.cz/en/people/francji1/01ZLMA.html) — Generalized Linear Models.

## Binary data: Bernoulli / Binomial GLM and logistic regression

This notebook covers **Chapter 6 of the lecture notes** (*Modely pro binární data*). Binary responses — dead / alive, survived / not, diseased / healthy — are the most common non-Gaussian use of GLMs and the template on which every other discrete-response model (Poisson regression, loglinear models) builds. The goal of this notebook is twofold:

1. **Theory.** Derive everything from scratch: the exponential-family form of the Bernoulli / Binomial distribution, the three standard link functions (logit, probit, cloglog), the log-likelihood, the score vector, the Fisher information matrix, the IRLS algorithm, the asymptotic distribution of the MLE, the deviance, and the three canonical tests — Wald, likelihood ratio, and Rao score.
2. **Practice.** Apply this machinery to two real datasets. **Titanic** (passenger survival) illustrates the interpretation of coefficients as log odds-ratios with discrete predictors and interactions. **Heart disease** (Cleveland clinical data) adds continuous predictors and serves as the vehicle for model diagnostics: Pearson and deviance residuals, binned residual plots, leverage, Cook's distance, DFBETAS, and the Hosmer–Lemeshow goodness-of-fit test.

The next exercise (ex08) goes further into prediction and model evaluation: ROC / AUC, calibration, cut-off choice, overdispersion, quasi-binomial fits, separation, and Firth's penalised likelihood.

## Contents

1. [Motivation — why a linear model is not enough](#1-motivation)
2. [Theoretical framework](#2-theory)
   - 2.1 [Bernoulli vs. Binomial paradigms](#21-bernoulli-vs-binomial)
   - 2.2 [Exponential family form](#22-exp-family)
   - 2.3 [Link functions](#23-link-functions)
   - 2.4 [Likelihood and log-likelihood](#24-likelihood)
   - 2.5 [Score vector](#25-score)
   - 2.6 [Fisher information matrix](#26-fisher-info)
   - 2.7 [IRLS algorithm](#27-irls)
   - 2.8 [Asymptotic distribution of the MLE](#28-asymptotics)
   - 2.9 [Deviance and saturated model](#29-deviance)
3. [Inference triangle: Wald, LRT, Rao](#3-inference)
4. [Odds ratios and model interpretation](#4-odds-ratios)
5. [Application 1 — Titanic (discrete predictors, OR)](#5-titanic)
6. [Application 2 — Heart disease (continuous predictors, diagnostics)](#6-heart)
7. [Your turn — tasks](#7-your-turn)
8. [Summary and transition to ex08](#8-summary)


## Setup

In [ ]:
import numpy as np
import pandas as pd

import scipy
from scipy import stats
from scipy.stats import chi2, norm

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.families.links import Logit, Probit, CLogLog
from statsmodels.stats.contingency_tables import Table2x2
from statsmodels.stats.outliers_influence import variance_inflation_factor

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import seaborn as sns
sns.set_theme(style="whitegrid")

import io
import os
import sys
import requests
import warnings
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

rng = np.random.default_rng(20260414)


In [ ]:
# Load helpers.py — works in Colab and locally.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q wget
    import wget
    wget.download(
        "https://github.com/francji1/01ZLMA/raw/main/code/helpers.py",
        "helpers.py",
    )
else:
    helpers_dir = os.path.dirname(os.path.abspath("__file__"))
    if helpers_dir not in sys.path:
        sys.path.insert(0, helpers_dir)

from helpers import Anova, DiagnosticPlots


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/francji1/01ZLMA/main/data/"

def load_csv(name, sep=","):
    # Fetch a CSV from the course GitHub repo in a Colab-friendly way.
    r = requests.get(BASE_URL + name, verify=False)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text), sep=sep)


---
# 1. Motivation — why a linear model is not enough <a id="1-motivation"></a>

Suppose the response is binary, $Y_i \in \{0, 1\}$. A linear model

$$Y_i = x_i^\top\beta + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, \sigma^2)$$

has at least three pathologies when the left-hand side is constrained to $\{0,1\}$:

1. **Prediction range.** $\hat\mu_i = x_i^\top\hat\beta$ can take any real value, so predicted "probabilities" can escape $[0,1]$.
2. **Heteroskedasticity.** For a Bernoulli response, $\mathrm{Var}(Y_i \mid x_i) = \pi_i(1-\pi_i)$ depends on the mean. OLS ignores this and reports mis-calibrated standard errors.
3. **Non-normal residuals.** With $Y_i \in \{0,1\}$ the residuals are bounded and bi-modal, violating every assumption of OLS inference.

The **generalised linear model** solves all three simultaneously by coupling a Bernoulli / Binomial response with a **link function** $g(\pi) = x^\top\beta$ that maps $[0,1]$ monotonically onto the real line. The canonical choice $g(\pi) = \log\frac{\pi}{1-\pi}$ gives the **logistic regression model** studied throughout this notebook.

Let us first see what OLS does on binary data:


In [ ]:
# Synthetic data: true probability is a logistic function of x.
n = 200
x = rng.uniform(-3, 3, n)
pi_true = 1 / (1 + np.exp(-(0.5 + 1.2 * x)))
y = rng.binomial(1, pi_true)

# Fit OLS and logistic regression, then plot both on the same grid.
df_demo = pd.DataFrame({"x": x, "y": y})
ols = smf.ols("y ~ x", data=df_demo).fit()
glm = smf.glm("y ~ x", data=df_demo, family=sm.families.Binomial()).fit()

xg = np.linspace(-4, 4, 400)
mu_ols = ols.params["Intercept"] + ols.params["x"] * xg
eta_glm = glm.params["Intercept"] + glm.params["x"] * xg
mu_glm = 1 / (1 + np.exp(-eta_glm))
mu_true = 1 / (1 + np.exp(-(0.5 + 1.2 * xg)))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(x, y, s=20, alpha=0.5, color="steelblue", label="binary data")
ax.plot(xg, mu_true, "k--", lw=1.2, label=r"true $\pi(x)$")
ax.plot(xg, mu_ols, color="tab:red", lw=2, label="OLS fit")
ax.plot(xg, mu_glm, color="tab:green", lw=2, label="logistic GLM")
ax.axhline(0, color="gray", lw=0.5); ax.axhline(1, color="gray", lw=0.5)
ax.set_ylim(-0.3, 1.3); ax.set_xlabel("x"); ax.set_ylabel(r"$\mathbb{E}[Y\mid x]$")
ax.legend(loc="center right"); ax.set_title("OLS on binary data: predictions escape [0,1]")
plt.tight_layout(); plt.show()


---
# 2. Theoretical framework <a id="2-theory"></a>

## 2.1 Bernoulli vs. Binomial paradigms <a id="21-bernoulli-vs-binomial"></a>

There are two equivalent ways of writing a GLM for binary data. They differ in how observations are grouped, not in the underlying likelihood.

**Bernoulli model** — each row is one binary trial:

$$Z_j \sim \mathrm{Be}(\pi_j), \qquad j = 1, \dots, N, \qquad \pi_j = g^{-1}(x_j^\top\beta).$$

**Binomial model** — when the regressor vectors $x_j$ take only $n$ distinct values ("covariate patterns"), the successes within each pattern can be aggregated:

$$Y_i = \sum_{j : x_j \in C_i} Z_j \sim \mathrm{Bi}(n_i, \pi_i), \qquad i = 1, \dots, n.$$

The two formulations give *identical* estimates $\hat\beta$ and deviances, because the Bernoulli likelihood factorises exactly into the binomial one. The Binomial form is more compact when data arrive as grouped counts (dose–response tables, contingency tables). The Bernoulli form is mandatory when some covariate is continuous, because every $x_j$ is typically unique.

**Asymptotic regimes.** The standard MLE asymptotics ($\hat\beta \to \beta$ at rate $\sqrt{n}$) requires one of:

- **Binomial regime:** $n$ (number of covariate patterns) is fixed, $n_i \to \infty$ for every pattern.
- **Bernoulli regime:** $n_i \approx 1$ for every $i$, and $N \to \infty$ with predictors that are not too collinear.

The two regimes have *different* goodness-of-fit behaviour: Pearson and deviance residuals, and the Hosmer–Lemeshow test, all assume enough replication within each covariate pattern, and they can be misleading under the Bernoulli regime.


## 2.2 Exponential family form <a id="22-exp-family"></a>

Recall (Exercise 01) that a distribution belongs to the one-parameter exponential family if its density can be written

$$f(y; \theta, \phi) = \exp\!\Big\{\frac{y\theta - b(\theta)}{a(\phi)} + c(y, \phi)\Big\}.$$

For $Y \sim \mathrm{Bi}(n, \pi)$ with $n$ known,

$$f(y; \pi) = \binom{n}{y}\pi^y(1-\pi)^{n-y}
   = \exp\Big\{y\log\tfrac{\pi}{1-\pi} + n\log(1-\pi) + \log\tbinom{n}{y}\Big\}.$$

Identifying terms,

$$\theta = \log\tfrac{\pi}{1-\pi},\qquad b(\theta) = n\log(1+e^\theta),\qquad a(\phi) = 1,\qquad c(y,\phi) = \log\tbinom{n}{y}.$$

From the exponential-family identities $\mathbb{E}[Y] = b'(\theta)$ and $\mathrm{Var}(Y) = b''(\theta)\,a(\phi)$:

$$b'(\theta) = \frac{n e^\theta}{1+e^\theta} = n\pi, \qquad b''(\theta) = n\pi(1-\pi).$$

Writing $\mu = \mathbb{E}[Y] = n\pi$, the **variance function** is

$$V(\mu) = \frac{\mu(n-\mu)}{n},$$

and for the scaled response $Y/n$ (which is what `statsmodels.GLM(..., family=Binomial())` uses when you pass `freq_weights=n`),

$$V(\pi) = \pi(1-\pi).$$

The **canonical link** is $\theta = \eta$, i.e. $g(\pi) = \log\tfrac{\pi}{1-\pi}$ — the logit.


## 2.3 Link functions <a id="23-link-functions"></a>

Any monotone $g : [0,1] \to \mathbb{R}$ could in principle serve as a link function. The practice is dominated by CDFs of four latent error distributions:

| Link        | $g(\pi)$                       | $g^{-1}(\eta)$                 | $\pi$ at $\eta = 0$     | Latent distribution        | Symmetric? |
|-------------|--------------------------------|-------------------------------|-------------------------|----------------------------|------------|
| **logit**   | $\log\frac{\pi}{1-\pi}$        | $\dfrac{e^\eta}{1+e^\eta}$    | $0.5$                   | Logistic$(0,1)$            | yes        |
| **probit**  | $\Phi^{-1}(\pi)$               | $\Phi(\eta)$                  | $0.5$                   | $\mathcal{N}(0,1)$         | yes        |
| **cloglog** | $\log(-\log(1-\pi))$           | $1 - \exp(-\exp\eta)$         | $1 - 1/e \approx 0.632$ | Gumbel (minimum, right-skewed) | no     |
| **loglog**  | $-\log(-\log\pi)$              | $\exp(-\exp(-\eta))$          | $1/e \approx 0.368$     | Gumbel (maximum, left-skewed)  | no     |

**Latent-variable interpretation.** Assume an unobserved $Y_i^\star = x_i^\top\beta + \varepsilon_i$ with $\varepsilon_i$ i.i.d. from some distribution $F$, and we observe $Y_i = \mathbb{1}[Y_i^\star > 0]$. Then $\pi_i = \mathbb{P}(Y_i = 1) = F(x_i^\top\beta)$, so $g = F^{-1}$. The four rows of the table correspond to four choices of $F$.

### The Gumbel distribution

Because it is far less familiar than the logistic or normal, recall the Gumbel law explicitly. The **Gumbel distribution of the minimum** with location $\mu$ and scale $s > 0$ has

$$F(x) = 1 - \exp\!\Big(-\exp\!\Big(\tfrac{x-\mu}{s}\Big)\Big), \qquad f(x) = \tfrac{1}{s}\,\exp\!\Big(\tfrac{x-\mu}{s}\Big)\exp\!\Big(-\exp\!\Big(\tfrac{x-\mu}{s}\Big)\Big),$$

with $\mathbb{E}[X] = \mu - \gamma s$, $\mathrm{Var}(X) = \pi^2 s^2 / 6$, where $\gamma \approx 0.5772$ is the Euler–Mascheroni constant. With $\mu = 0$, $s = 1$ the CDF is $F(x) = 1 - \exp(-\exp x)$ — setting $\pi = F(\eta)$ gives exactly the cloglog inverse link. The mirror-image **Gumbel distribution of the maximum** has CDF $F(x) = \exp(-\exp(-x))$ and corresponds to the loglog link.

The Gumbel distribution arises as the asymptotic distribution of minima / maxima of i.i.d. samples (Fisher–Tippett–Gnedenko theorem) and is used extensively in extreme-value analysis, discrete-time survival models, and choice theory.

### Why asymmetry matters

**cloglog** and **loglog** are mirror images of each other under $\pi \leftrightarrow 1-\pi$: if $Y$ follows cloglog with $\pi = 1 - \exp(-\exp\eta)$, then $1 - Y$ follows loglog with $1 - \pi = \exp(-\exp\eta)$. They approach the two endpoints at *different rates*:

- **cloglog** rises steeply near $\pi = 1$: the curve passes through $0.632$ at $\eta = 0$, spending more of its range in $(0.5, 1)$. Natural for "rare-non-event" settings where almost everyone experiences the outcome unless specifically protected (discrete hazard models: $\pi = 1 - \exp(-\lambda),\ \lambda = e^{x^\top\beta}$).
- **loglog** rises steeply near $\pi = 0$: the curve passes through $0.368$ at $\eta = 0$. Natural for "rare-event" settings where the baseline probability is very small.
- **logit** and **probit** are indistinguishable for $\pi \in (0.1, 0.9)$ and differ from each other only by a scale factor $\approx \pi/\sqrt{3} \approx 1.81$ (see solutions notebook T4).

**Practical rule.** Start with the logit (canonical, interpretable as log-odds, numerically stable). Switch to probit when a normal latent scale is physically meaningful (thresholded sensory measurements). Switch to cloglog for rare-non-event / constant-hazard settings; use loglog for the mirror case.


In [ ]:
# Plot the four link functions on a common grid, both as g(pi) and as the
# inverse link (the CDF of the latent error distribution).
pi_grid  = np.linspace(1e-3, 1 - 1e-3, 500)
eta_grid = np.linspace(-5, 5, 500)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.5))

# Left panel: g(pi) = linear predictor eta as a function of pi.
axL.plot(pi_grid, np.log(pi_grid / (1 - pi_grid)),     label="logit",   lw=2)
axL.plot(pi_grid, norm.ppf(pi_grid),                   label="probit",  lw=2)
axL.plot(pi_grid, np.log(-np.log(1 - pi_grid)),        label="cloglog", lw=2)
axL.plot(pi_grid, -np.log(-np.log(pi_grid)),           label="loglog",  lw=2, ls="--")
axL.axhline(0, color="gray", lw=0.5); axL.axvline(0.5, color="gray", lw=0.5)
axL.set_xlabel(r"$\pi$"); axL.set_ylabel(r"$g(\pi)$")
axL.set_title(r"Link functions $g(\pi) = \eta$")
axL.legend()

# Right panel: inverse link — pi as a function of eta. This is the CDF of
# the latent error, and shows the asymmetry of cloglog / loglog clearly.
axR.plot(eta_grid, 1 / (1 + np.exp(-eta_grid)),        label="logit",   lw=2)
axR.plot(eta_grid, norm.cdf(eta_grid),                  label="probit",  lw=2)
axR.plot(eta_grid, 1 - np.exp(-np.exp(eta_grid)),       label="cloglog", lw=2)
axR.plot(eta_grid, np.exp(-np.exp(-eta_grid)),          label="loglog",  lw=2, ls="--")
axR.axhline(0.5, color="gray", lw=0.5); axR.axvline(0, color="gray", lw=0.5)
# Mark the non-symmetric crossings at eta = 0.
axR.plot(0, 1 - 1/np.e, "o", color="tab:green",  ms=7)
axR.annotate(r"cloglog: $\pi\!\approx\!0.632$", xy=(0, 1 - 1/np.e),
             xytext=(0.3, 0.82), fontsize=9, color="tab:green")
axR.plot(0, 1/np.e,       "o", color="tab:red",   ms=7)
axR.annotate(r"loglog: $\pi\!\approx\!0.368$", xy=(0, 1/np.e),
             xytext=(0.3, 0.22), fontsize=9, color="tab:red")
axR.set_xlabel(r"$\eta$"); axR.set_ylabel(r"$\pi = g^{-1}(\eta)$")
axR.set_title(r"Inverse link $g^{-1}(\eta) = \pi$ (= CDF of latent error)")
axR.legend(loc="center right")

plt.tight_layout(); plt.show()


## 2.4 Likelihood and log-likelihood <a id="24-likelihood"></a>

Write the binomial log-likelihood as a function of $\beta$. For the Binomial model $Y_i \sim \mathrm{Bi}(n_i, \pi_i)$ with $\pi_i = g^{-1}(x_i^\top\beta)$,

$$\ell(\beta) = \sum_{i=1}^n \Big[y_i \log\pi_i + (n_i - y_i)\log(1 - \pi_i) + \log\tbinom{n_i}{y_i}\Big].$$

The last term does not depend on $\beta$ and drops out for inference. For the **canonical (logit)** link, the identity $\log\frac{\pi_i}{1-\pi_i} = x_i^\top\beta$ lets us rewrite

$$\ell(\beta) = \sum_{i=1}^n \Big[y_i\, x_i^\top\beta \;-\; n_i \log\!\big(1 + e^{x_i^\top\beta}\big)\Big] \;+\; \text{const}. \tag{2.4.1}$$

This is a concave function of $\beta$ (Exercise below), so the MLE is either unique or fails to exist (in the case of perfect separation). No local maxima.


## 2.5 Score vector <a id="25-score"></a>

Differentiate $\ell(\beta)$ in (2.4.1) coordinate-wise:

$$\frac{\partial \ell}{\partial \beta_k}
   = \sum_{i=1}^n \Big[y_i\, x_{ik} - n_i\, x_{ik}\, \frac{e^{x_i^\top\beta}}{1 + e^{x_i^\top\beta}}\Big]
   = \sum_{i=1}^n (y_i - n_i\pi_i)\, x_{ik}.$$

Stacking $k = 1, \dots, p$:

$$\boxed{\;U(\beta) \;=\; \frac{\partial \ell}{\partial \beta} \;=\; X^\top (y - \mu), \qquad \mu_i = n_i\pi_i.\;}$$

The beautiful form — residuals times design — is specific to the *canonical* link. For the probit or cloglog links the score is

$$U(\beta) = \sum_i \frac{y_i - n_i\pi_i}{\pi_i(1-\pi_i)}\; g'(\pi_i)^{-1}\, x_i = X^\top W\, \tilde r,$$

with $\tilde r_i = (y_i - n_i\pi_i)\,\big/\big(n_i\pi_i(1-\pi_i)\,g'(\pi_i)\big)$ and weights $W_{ii}$ derived below. The canonical form comes out because $g'(\pi) = 1/(\pi(1-\pi))$ exactly cancels the variance function.


## 2.6 Fisher information matrix <a id="26-fisher-info"></a>

The Fisher information is the expected negative Hessian, $\mathcal{I}(\beta) = -\mathbb{E}[\partial^2 \ell / \partial\beta\partial\beta^\top]$. Differentiating the score once more,

$$\frac{\partial^2 \ell}{\partial\beta_k \partial\beta_\ell}
   = -\sum_i n_i \pi_i(1-\pi_i)\, x_{ik} x_{i\ell},$$

which is non-random (no $y_i$), so the observed Hessian equals the expected Hessian. For the canonical link,

$$\boxed{\;\mathcal{I}(\beta) \;=\; X^\top W X, \qquad W = \mathrm{diag}\!\big(n_i \pi_i (1-\pi_i)\big).\;}$$

For non-canonical links the general formula is $W_{ii} = (d\mu_i/d\eta_i)^2 / V(\mu_i)$; this is what `helpers.Anova`'s Rao test uses internally.

**Consequence.** The asymptotic covariance of the MLE is $(X^\top W X)^{-1}$. With binary data and no separation, $W$ is strictly positive definite and the covariance is well-defined.


## 2.7 IRLS algorithm <a id="27-irls"></a>

Newton–Raphson iteration for $\ell$:

$$\beta^{(t+1)} \;=\; \beta^{(t)} + \big[\mathcal{I}(\beta^{(t)})\big]^{-1} U(\beta^{(t)}).$$

Substitute $\mathcal{I} = X^\top W X$ and $U = X^\top(y - \mu)$, multiply through by $X^\top W X$, and massage:

$$(X^\top W X)\, \beta^{(t+1)}
   \;=\; (X^\top W X)\,\beta^{(t)} + X^\top(y - \mu^{(t)})
   \;=\; X^\top W \Big(\underbrace{X\beta^{(t)} + W^{-1}(y - \mu^{(t)})}_{=:\, z^{(t)}}\Big).$$

So each Newton step is a **weighted least squares fit** of the **working response** $z^{(t)}$ on $X$:

$$\boxed{\;\beta^{(t+1)} = (X^\top W^{(t)} X)^{-1} X^\top W^{(t)} z^{(t)}, \qquad
   z_i^{(t)} = \eta_i^{(t)} + \frac{y_i - n_i\pi_i^{(t)}}{n_i\pi_i^{(t)}(1-\pi_i^{(t)})}.\;}$$

Hence the name **iteratively reweighted least squares**. For binary GLM with the canonical link, IRLS converges in ~5–10 iterations for non-separated data. The algorithm below is a 30-line implementation; we verify it against `statsmodels.GLM`.


In [ ]:
def irls_binary(X, y, n=None, tol=1e-8, max_iter=100):
    # IRLS for the binomial GLM with canonical (logit) link.
    # X : (N, p) design matrix (include the intercept column explicitly).
    # y : (N,) successes; n : (N,) trials (all ones = Bernoulli).
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    if n is None:
        n = np.ones_like(y)
    n = np.asarray(n, dtype=float)

    # Start at the mean response.
    pi = np.clip((y + 0.5) / (n + 1), 1e-3, 1 - 1e-3)
    eta = np.log(pi / (1 - pi))
    beta = np.linalg.lstsq(X, eta, rcond=None)[0]

    history = []
    for t in range(max_iter):
        eta = X @ beta
        pi = 1 / (1 + np.exp(-eta))
        mu = n * pi
        W = n * pi * (1 - pi)                               # IRLS weights
        z  = eta + (y - mu) / W                             # working response
        XtWX = (X * W[:, None]).T @ X
        XtWz = (X * W[:, None]).T @ z
        beta_new = np.linalg.solve(XtWX, XtWz)
        diff = np.max(np.abs(beta_new - beta))
        history.append(diff)
        beta = beta_new
        if diff < tol:
            break
    cov = np.linalg.inv((X * (n * pi * (1 - pi))[:, None]).T @ X)
    return beta, cov, history


# Simulated data with known beta.
beta_true = np.array([-1.0, 0.8, -0.5])
Nsim = 400
X = np.column_stack([np.ones(Nsim), rng.normal(size=Nsim), rng.normal(size=Nsim)])
pi = 1 / (1 + np.exp(-(X @ beta_true)))
ys = rng.binomial(1, pi)

beta_hat, cov_hat, hist = irls_binary(X, ys)
glm_fit = sm.GLM(ys, X, family=sm.families.Binomial()).fit()

print("True beta   :", beta_true)
print("IRLS  beta  :", np.round(beta_hat, 6))
print("GLM   beta  :", np.round(glm_fit.params, 6))
print("Max abs diff:", np.max(np.abs(beta_hat - glm_fit.params)))
print(f"Converged in {len(hist)} iterations (‖Δβ‖∞ history):")
print("  ", [f"{h:.2e}" for h in hist])


## 2.8 Asymptotic distribution of the MLE <a id="28-asymptotics"></a>

Under standard regularity conditions (differentiability of $\ell$, invertibility of $\mathcal{I}(\beta)$, and enough observations per covariate pattern),

$$\sqrt{N}\,(\hat\beta - \beta) \xrightarrow{d} \mathcal{N}\!\big(0, \; N\,\mathcal{I}(\beta)^{-1}\big),$$

or equivalently $\hat\beta \,\dot\sim\, \mathcal{N}\big(\beta, (X^\top W X)^{-1}\big)$. For the binary GLM, **$N$ means "effective sample size"**: in the Bernoulli regime $N$ is the number of observations; in the Binomial regime $N = \sum_i n_i$.

This justifies all three classical tests covered in Section 3. The Monte Carlo simulation in the companion solutions notebook shows how fast (or slow) this convergence actually is — for small $N$ the Wald test in particular has poor coverage.


## 2.9 Deviance and saturated model <a id="29-deviance"></a>

The **saturated model** has one parameter per observation ($\hat\pi_i^{\text{sat}} = y_i / n_i$) and therefore achieves the maximum possible log-likelihood, $\ell_{\text{sat}}$. For any fitted model $M$ with log-likelihood $\ell(\hat\beta; y)$, the **deviance**

$$D(y; \hat\mu) \;=\; 2\big[\ell_{\text{sat}} - \ell(\hat\beta; y)\big]$$

measures how far $M$ is from a perfect fit. For the binomial model,

$$D \;=\; 2 \sum_{i=1}^n \Big[y_i \log\!\tfrac{y_i}{\hat\mu_i} + (n_i - y_i)\log\!\tfrac{n_i - y_i}{n_i - \hat\mu_i}\Big], \qquad \hat\mu_i = n_i\hat\pi_i.$$

With the convention $0\log 0 = 0$, $D \geq 0$ with equality iff $\hat\mu_i = y_i$ for all $i$.

For two nested models $M_0 \subset M$,

$$D_0 - D \;\xrightarrow{d}\; \chi^2_{p - p_0} \quad\text{under } H_0 : M_0 \text{ holds},$$

which is precisely the LRT statistic of Section 3.2. Under the Binomial regime (enough replicates per pattern), $D \sim \chi^2_{n - p}$ under $M$, yielding a **goodness-of-fit** test: if $D$ is much larger than its $\chi^2_{n - p}$ quantile, the model does not fit. In the Bernoulli regime this interpretation breaks down — $D$ does not converge in distribution, and one must fall back to the Hosmer–Lemeshow test (Section 6).


---
# 3. Inference triangle: Wald, LRT, Rao <a id="3-inference"></a>

Three large-sample tests coexist for the hypothesis $H_0 : \beta_1 = 0$ with $\beta = (\beta_0, \beta_1)$. They are asymptotically equivalent (all $\to \chi^2_{p_1}$ under $H_0$) but differ in what needs to be fitted:

| Test | What is fitted?                 | Statistic                                                 |
|------|---------------------------------|-----------------------------------------------------------|
| **Wald**  | The *full* model $M$.                  | $W = \hat\beta_1^\top\, \widehat{\mathrm{Cov}}(\hat\beta_1)^{-1}\, \hat\beta_1$    |
| **LRT**   | *Both* $M_0$ and $M$.                 | $T_1 = 2(\ell_M - \ell_{M_0}) = (D_0 - D)/\phi$             |
| **Rao**   | Only the *null* model $M_0$.          | $T_R = U(\hat\beta_0)^\top \mathcal{I}(\hat\beta_0)^{-1} U(\hat\beta_0) / \phi$  |

For the Binomial / Bernoulli family $\phi = 1$.


## 3.1 Wald test <a id="31-wald"></a>

Under $H_0$,

$$W \;=\; \hat\beta_1^\top\, \widehat{\mathrm{Cov}}(\hat\beta_1)^{-1}\, \hat\beta_1 \;\dot\sim\; \chi^2_{p_1}.$$

The Wald statistic is what `statsmodels.GLM` reports next to each coefficient. Its main appeal is computational: only one fit needed. Its drawback is the **Hauck–Donner effect** — for well-separated data the estimated standard error $\sqrt{\widehat{\mathrm{Cov}}(\hat\beta_1)}$ *grows* when $\hat\beta_1$ moves farther from zero, so $W$ can *decrease* as the evidence against $H_0$ *increases*. The LRT and Rao tests do not suffer from this.


## 3.2 Likelihood ratio test (LRT) <a id="32-lrt"></a>

$$T_1 \;=\; 2\big(\ell(\hat\beta; M) - \ell(\hat\beta_0; M_0)\big) \;=\; \frac{D_0 - D}{\phi} \;\dot\sim\; \chi^2_{p_1}.$$

The LRT is widely preferred in practice: it has better small-sample behaviour than the Wald test, uses the actual likelihood shape rather than a local quadratic approximation, and is invariant to reparametrisation. Its one cost is the need to fit *both* models.

In R this is `anova(m0, m, test = "Chisq")`. With `helpers.Anova` we do the same:


## 3.3 Rao score test <a id="33-rao"></a>

$$T_R \;=\; U(\hat\beta_0)^\top \mathcal{I}(\hat\beta_0)^{-1} U(\hat\beta_0) / \phi \;\dot\sim\; \chi^2_{p_1},$$

where $U$ and $\mathcal{I}$ are the score and *expected* Fisher information of the *full* model $M$, evaluated at the null estimate $\hat\beta_0$ extended with zeros for the extra coordinates of $M$. The key advantage: only the *null* model is ever fit. That makes Rao the natural test for many quick screening settings (e.g. variable screening where we would need $p$ full-model fits for a Wald loop).

*Caveat on the implementation in `helpers.py`:* the Rao test uses the expected Fisher information, and for non-canonical links this is not identical to the observed Hessian. Numerical discrepancies against R's `anova.glm(..., test = "Rao")` are typically below $10^{-8}$; see the docstring of `helpers.Anova`.


## 3.4 Which test when? <a id="34-when"></a>

| Situation                                                              | Prefer        |
|------------------------------------------------------------------------|---------------|
| Standard coefficient $p$-value next to `.summary()`                     | Wald (free)   |
| Compare two nested fitted models                                       | **LRT**       |
| Screen many candidate variables without fitting each one               | Rao           |
| Suspected near-separation or very large coefficients                    | LRT (Wald unreliable) |
| Goodness-of-fit against saturated model (Binomial regime, $n_i$ large) | LRT or Rao    |

All three tests are asymptotically equivalent under $H_0$ and under *local* alternatives $\beta_1 = h / \sqrt{N}$. They differ under distant alternatives and in small samples — the solutions notebook contains a Monte Carlo study of this.


In [ ]:
# Demonstrate the three tests on a single example: full model vs. null.
N = 250
demo = pd.DataFrame({"x1": rng.normal(size=N), "x2": rng.normal(size=N)})
eta  = -0.3 + 0.7 * demo["x1"] - 0.4 * demo["x2"]
demo["y"] = rng.binomial(1, 1 / (1 + np.exp(-eta)))

m_full = smf.glm("y ~ x1 + x2", data=demo, family=sm.families.Binomial()).fit()
m_null = smf.glm("y ~ 1",       data=demo, family=sm.families.Binomial()).fit()

anova = Anova()
print("=== LRT ===");  print(anova(m_null, m_full, test="LRT",  verbose=False)); print()
print("=== Wald ==="); print(anova(m_null, m_full, test="Wald", verbose=False)); print()
print("=== Rao ===");  print(anova(m_null, m_full, test="Rao",  verbose=False))


---
# 4. Odds ratios and model interpretation <a id="4-odds-ratios"></a>

For logistic regression, coefficients are *directly* interpretable once exponentiated. That makes the logit link the default choice whenever interpretability matters.

**Odds.** $\mathrm{odds}(\pi) = \pi / (1 - \pi)$, ranging over $(0, \infty)$. Odds of 1 means $\pi = 0.5$; odds of 3 means "three times as likely to succeed as to fail".

**Odds ratio.** For two values $x_a, x_b$ of a single predictor with other predictors held fixed,

$$\mathrm{OR}(x_a, x_b) \;=\; \frac{\pi(x_a) / (1 - \pi(x_a))}{\pi(x_b) / (1 - \pi(x_b))}.$$

In a logit model $\log\frac{\pi(x)}{1-\pi(x)} = x^\top\beta$:

- For a **binary** predictor with $\beta_j$ its coefficient, $\mathrm{OR}(x_j=1, x_j=0) = e^{\beta_j}$.
- For a **continuous** predictor, $\mathrm{OR}(x_j + c, x_j) = e^{c\beta_j}$ — per $c$ units.
- For a **categorical** predictor with reference category, $\mathrm{OR}(\text{level } k, \text{reference}) = e^{\beta_k}$.

**CI for the OR.** Because $\log \mathrm{OR} = \beta_j$ is asymptotically normal, a clean Wald interval is

$$\big[e^{\hat\beta_j - z_{1-\alpha/2}\,\mathrm{se}(\hat\beta_j)}, \; e^{\hat\beta_j + z_{1-\alpha/2}\,\mathrm{se}(\hat\beta_j)}\big].$$

**Interactions.** In a model with $\beta_j x_j + \beta_k x_k + \beta_{jk} x_j x_k$, the OR for $x_j$ *depends on the value of $x_k$*: $\mathrm{OR}(x_j + 1, x_j \mid x_k) = \exp(\beta_j + \beta_{jk} x_k)$. Marginal ORs in the presence of interactions are typically not what one wants — report the interaction term or conditional ORs at chosen $x_k$.


In [ ]:
def OR_ci(model, var_name, alpha=0.05):
    # Odds ratio and Wald CI for a single parameter.
    b  = model.params[var_name]
    se = model.bse[var_name]
    z  = norm.ppf(1 - alpha / 2)
    return pd.Series({
        "OR":     np.exp(b),
        "CI_low": np.exp(b - z * se),
        "CI_up":  np.exp(b + z * se),
        "p_val":  model.pvalues[var_name],
    })


---
# 5. Application 1 — Titanic <a id="5-titanic"></a>

The Titanic passenger list is the textbook dataset for illustrating logistic regression with discrete predictors. We load it from seaborn (which ships a clean cached copy) to keep the notebook self-contained. The binary response is `survived` $\in \{0, 1\}$; predictors of interest are `sex`, `pclass` (1 / 2 / 3), `embarked` (C / Q / S), and later the continuous `age` and `fare`.


In [ ]:
titanic = sns.load_dataset("titanic").copy()
print(titanic.shape)
titanic.head()


In [ ]:
# Keep only what we need; drop rows with missing values in the discrete subset.
data_dis = titanic[["survived", "pclass", "sex", "embarked"]].dropna().copy()
for c in ["pclass", "sex", "embarked"]:
    data_dis[c] = data_dis[c].astype("category")
print(data_dis.dtypes)
print("\nBaseline survival rate:", data_dis["survived"].mean().round(3))
data_dis.head()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, col in zip(axes, ["sex", "pclass", "embarked"]):
    sns.countplot(data=data_dis, x=col, hue="survived", ax=ax, palette="Set2")
    ax.set_title(f"survived by {col}")
plt.tight_layout(); plt.show()


## 5.1 Null model → sex → pclass → interactions <a id="51-sex-pclass"></a>

We build a sequence of nested models and run the LRT at each step.

In [ ]:
m0   = smf.glm("survived ~ 1",                            data=data_dis, family=sm.families.Binomial()).fit()
m_s  = smf.glm("survived ~ sex",                          data=data_dis, family=sm.families.Binomial()).fit()
m_sp = smf.glm("survived ~ sex + pclass",                 data=data_dis, family=sm.families.Binomial()).fit()
m_se = smf.glm("survived ~ sex + pclass + embarked",      data=data_dis, family=sm.families.Binomial()).fit()
m_i  = smf.glm("survived ~ sex * pclass + embarked",      data=data_dis, family=sm.families.Binomial()).fit()

anova(m0, m_s, m_sp, m_se, m_i, test="LRT")


In [ ]:
# All odds ratios for the additive model with pclass + sex + embarked.
ors = pd.concat({v: OR_ci(m_se, v) for v in m_se.params.index if v != "Intercept"}, axis=1).T
ors.round(3)


**Reading the output.**

- `sex[T.male]` has OR $\approx 0.08$: male passengers had about $1/12$ the odds of survival of female passengers (all else equal). Equivalently, women had $\sim 12\times$ higher odds.
- `pclass[T.3]` has OR well below 1: third-class passengers had dramatically lower survival odds relative to first class.
- `embarked` is borderline; the LRT above tells us whether it adds explanatory power given `sex + pclass`.

The saturated-in-discrete model `m_i` adds a `sex:pclass` interaction — the LRT tells us whether the sex effect differs by class.


In [ ]:
# Directly interpret the interaction: OR for being male, by class.
coef = m_i.params
or_male_c1 = np.exp(coef["sex[T.male]"])
or_male_c2 = np.exp(coef["sex[T.male]"] + coef["sex[T.male]:pclass[T.2]"])
or_male_c3 = np.exp(coef["sex[T.male]"] + coef["sex[T.male]:pclass[T.3]"])
print(f"OR(male vs female) in pclass 1: {or_male_c1:.3f}")
print(f"OR(male vs female) in pclass 2: {or_male_c2:.3f}")
print(f"OR(male vs female) in pclass 3: {or_male_c3:.3f}")


## 5.2 Continuous predictor: age and fare <a id="52-age-fare"></a>

In [ ]:
data_con = titanic[["survived", "pclass", "sex", "embarked", "age", "fare"]].dropna().copy()
for c in ["pclass", "sex", "embarked"]:
    data_con[c] = data_con[c].astype("category")

# age in decades so the coefficient has an interpretable scale.
data_con["age10"] = data_con["age"] / 10

m_age  = smf.glm("survived ~ age10",                                 data=data_con, family=sm.families.Binomial()).fit()
m_full = smf.glm("survived ~ sex + pclass + embarked + age10 + fare", data=data_con, family=sm.families.Binomial()).fit()
anova(m_age, m_full, test="LRT")


In [ ]:
OR_ci(m_full, "age10").round(3)


The OR for `age10` is interpreted as "odds ratio of surviving per 10 years of additional age". An estimated OR of $\approx 0.6$ — e.g. — says that every decade of extra age cut the odds of survival by about $40\%$, *holding sex, class, port of embarkation, and fare fixed*. Causality is a separate question; here we are merely describing a conditional association in the observed data.


---
# 6. Application 2 — Heart disease (continuous predictors, diagnostics) <a id="6-heart"></a>

The Cleveland Heart Disease dataset (available as `heart_train.csv` + `heart_test.csv` in the course data folder) records clinical measurements for 303 patients; the binary response `disease` is 1 if the patient was diagnosed with heart disease. Predictors include a mix of continuous (`age`, `blood_pressure`, `cholesterol`, `heart_rate`, `st_depression`) and categorical (`sex`, `chest_pain_type`, `blood_sugar`, `ex_angina`, `st_slope`, `num_vessels`, `thal`) variables — ideal for illustrating full model diagnostics.


In [ ]:
heart = load_csv("heart_train.csv")
for c in ["sex", "chest_pain_type", "blood_sugar", "rest_ecg", "ex_angina", "st_slope", "num_vessels", "thal"]:
    heart[c] = heart[c].astype("category")

print(heart.shape)
print(heart.dtypes)
heart.head()


In [ ]:
# Prevalence of the response.
heart["disease"].value_counts(normalize=True).round(3)


## 6.1 Link-function comparison: logit vs. probit vs. cloglog <a id="61-link-comparison"></a>

Fit the same model with three different links and compare:

1. **Coefficients are not directly comparable** — they live on different scales.
2. **Fitted probabilities** should be nearly identical, especially away from 0 and 1.
3. **Deviance** differs and is one (small) diagnostic for which link fits better.


In [ ]:
formula = ("disease ~ age + sex + chest_pain_type + blood_pressure + cholesterol "
           "+ blood_sugar + heart_rate + ex_angina + st_depression + st_slope")

m_logit   = smf.glm(formula, data=heart, family=sm.families.Binomial(link=Logit())).fit()
m_probit  = smf.glm(formula, data=heart, family=sm.families.Binomial(link=Probit())).fit()
m_cloglog = smf.glm(formula, data=heart, family=sm.families.Binomial(link=CLogLog())).fit()

pd.DataFrame({
    "link":     ["logit",          "probit",           "cloglog"],
    "deviance": [m_logit.deviance, m_probit.deviance,  m_cloglog.deviance],
    "AIC":      [m_logit.aic,      m_probit.aic,       m_cloglog.aic],
    "df_resid": [m_logit.df_resid, m_probit.df_resid,  m_cloglog.df_resid],
}).round(3)


In [ ]:
# Fitted probabilities: three links vs. each other.
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(m_logit.fittedvalues, m_probit.fittedvalues,  s=18, alpha=0.6, color="tab:orange")
ax[0].plot([0, 1], [0, 1], "k--", lw=0.8); ax[0].set_xlabel("logit fit"); ax[0].set_ylabel("probit fit")
ax[0].set_title("probit vs. logit (fitted prob.)")

ax[1].scatter(m_logit.fittedvalues, m_cloglog.fittedvalues, s=18, alpha=0.6, color="tab:green")
ax[1].plot([0, 1], [0, 1], "k--", lw=0.8); ax[1].set_xlabel("logit fit"); ax[1].set_ylabel("cloglog fit")
ax[1].set_title("cloglog vs. logit (fitted prob.)")
plt.tight_layout(); plt.show()


## 6.2 Residuals: Pearson, deviance, binned <a id="62-residuals"></a>

For the binomial model the two standard residuals are

$$r_i^P = \frac{y_i - n_i\hat\pi_i}{\sqrt{n_i\hat\pi_i(1-\hat\pi_i)}}\quad\text{(Pearson)},\qquad
 r_i^D = \mathrm{sign}(y_i - n_i\hat\pi_i)\,\sqrt{2\big[y_i\log\tfrac{y_i}{n_i\hat\pi_i} + (n_i-y_i)\log\tfrac{n_i-y_i}{n_i(1-\hat\pi_i)}\big]}\quad\text{(deviance)}.$$

Under the **Binomial regime** with $\min\{y_i, n_i - y_i\} \geq 3$, both have approximately $\mathcal{N}(0, 1)$ distribution and the usual residual plots apply. Under the **Bernoulli regime** ($n_i = 1$), Pearson residuals take only two values per observation and scatter plots show two stripes — the standard remedy is **binned residuals** (Gelman & Hill): sort by fitted value, split into bins, and plot the *mean* raw residual per bin. A well-fitting model should have most binned residuals within $\pm 2/\sqrt{n_{\text{bin}}}$ of zero.


In [ ]:
def binned_residuals(model, n_bins=30):
    # Binned residual plot (Gelman & Hill 2006, section 5.3).
    # Sort observations by fitted probability, split into n_bins equal-count bins,
    # plot the mean raw residual against the mean fitted probability per bin.
    # The +/- 2/sqrt(n_bin) band is a rough 95% pointwise reference.
    p_hat = np.asarray(model.fittedvalues)
    raw   = np.asarray(model.resid_response)
    order = np.argsort(p_hat)
    p_hat, raw = p_hat[order], raw[order]
    edges = np.linspace(0, len(p_hat), n_bins + 1).astype(int)
    centres, means, bounds = [], [], []
    for a, b in zip(edges[:-1], edges[1:]):
        if b - a < 2:
            continue
        centres.append(p_hat[a:b].mean())
        means.append(raw[a:b].mean())
        bounds.append(2 * raw[a:b].std(ddof=1) / np.sqrt(b - a))
    return np.array(centres), np.array(means), np.array(bounds)


centres, means, bounds = binned_residuals(m_logit, n_bins=20)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
# Pearson residuals vs. fitted — Bernoulli stripes visible.
ax[0].scatter(m_logit.fittedvalues, m_logit.resid_pearson, s=16, alpha=0.5)
ax[0].axhline(0, color="k", lw=0.5)
ax[0].set_xlabel("fitted probability"); ax[0].set_ylabel("Pearson residual")
ax[0].set_title("Pearson residuals (Bernoulli regime: two stripes)")
# Binned residuals.
ax[1].plot(centres, means, "o-", color="tab:red")
ax[1].plot(centres, bounds, ":", color="gray"); ax[1].plot(centres, -bounds, ":", color="gray")
ax[1].axhline(0, color="k", lw=0.5)
ax[1].set_xlabel("mean fitted probability (bin)"); ax[1].set_ylabel("mean raw residual")
ax[1].set_title("Binned residuals (20 bins)")
plt.tight_layout(); plt.show()


## 6.3 Leverage, Cook's distance, DFBETAS <a id="63-influence"></a>

The projection (hat) matrix for a GLM is $H = W^{1/2} X (X^\top W X)^{-1} X^\top W^{1/2}$, with $W$ the IRLS weights. Its diagonal $h_{ii}$ plays the same role as in OLS: how much the $i$-th observation determines its own fitted value. Cook's distance for a GLM,

$$\mathrm{CD}_i = \frac{1}{p}\,(r_i^{PS})^2 \, \frac{h_{ii}}{1 - h_{ii}}, \qquad r_i^{PS} = \frac{r_i^P}{\sqrt{1 - h_{ii}}},$$

flags observations with both high leverage and a large standardised Pearson residual. A common rule of thumb is $\mathrm{CD}_i > 8 / (n - 2p)$.

`helpers.DiagnosticPlots` wraps these computations:


In [ ]:
dp = DiagnosticPlots(m_logit)
fig, ax = dp(plot_context="seaborn-v0_8-paper")
plt.show()


In [ ]:
# Top 5 observations by Cook's distance.
infl = m_logit.get_influence()
cook = infl.cooks_distance[0]
top = np.argsort(cook)[::-1][:5]
pd.DataFrame({
    "row":    top,
    "CookD":  cook[top],
    "leverage": infl.hat_matrix_diag[top],
    "resid_pearson_std": m_logit.resid_pearson[top] / np.sqrt(1 - infl.hat_matrix_diag[top]),
    "y":      heart["disease"].iloc[top].values,
    "fitted": m_logit.fittedvalues.iloc[top].values,
}).round(3)


## 6.4 Hosmer–Lemeshow goodness-of-fit test <a id="64-hl-test"></a>

Under the Bernoulli regime, the residual deviance no longer has an asymptotic $\chi^2$ distribution, so the standard `D / df_resid` rule fails. Hosmer & Lemeshow (1980) proposed:

1. Rank observations by the fitted probability $\hat\pi_i$.
2. Split them into $G$ groups of (approximately) equal size. A common choice is $G = 10$ (deciles of risk).
3. In each group $g$, compare observed successes $Y_g = \sum y_i$ against expected $E_g = \sum \hat\pi_i$:

$$\hat{C} \;=\; \sum_{g=1}^G \frac{(Y_g - E_g)^2}{E_g\,(1 - E_g / n_g)} \;\dot\sim\; \chi^2_{G-2}.$$

A large $\hat{C}$ (small $p$-value) signals lack of fit. The $G - 2$ degrees of freedom were determined empirically by the original authors; some implementations use $G - 1$ or $G - p$.


In [ ]:
def hosmer_lemeshow(model, g=10):
    # Hosmer-Lemeshow C-hat goodness-of-fit test.
    # Returns: chi^2 statistic, degrees of freedom, p-value, and the 2xG table
    # of observed vs. expected successes per decile of fitted risk.
    p_hat = np.asarray(model.fittedvalues)
    y     = np.asarray(model.model.endog)
    # Equal-count bins; use pd.qcut to handle ties in fitted probabilities.
    bins = pd.qcut(p_hat, q=g, labels=False, duplicates="drop")
    G    = bins.max() + 1
    obs_1, exp_1, n_g = [], [], []
    for k in range(G):
        mask = bins == k
        n_g.append(mask.sum())
        obs_1.append(y[mask].sum())
        exp_1.append(p_hat[mask].sum())
    obs_1, exp_1, n_g = map(np.asarray, (obs_1, exp_1, n_g))
    stat = np.sum((obs_1 - exp_1) ** 2 / (exp_1 * (1 - exp_1 / n_g)))
    df   = G - 2
    p    = 1 - chi2.cdf(stat, df)
    table = pd.DataFrame({
        "n":      n_g,
        "obs_1":  obs_1,
        "exp_1":  exp_1.round(2),
        "obs_0":  n_g - obs_1,
        "exp_0":  (n_g - exp_1).round(2),
    })
    return stat, df, p, table


stat, df, p, tbl = hosmer_lemeshow(m_logit, g=10)
print(f"Hosmer-Lemeshow C-hat = {stat:.3f}  on  df = {df}   p = {p:.3f}")
tbl


## 6.5 A note on separation <a id="65-separation"></a>

If a covariate (or a linear combination of covariates) *perfectly* separates the two classes, the MLE of the corresponding coefficient does not exist — it diverges to $\pm\infty$, and `statsmodels` typically warns about *PerfectSeparation* or silently returns a huge estimate with a giant standard error.

**Quasi-complete separation** is subtler: $\pi_i = 0$ or $1$ holds for some observations, typically within a thin subregion of the covariate space. The warning signs are:

- Coefficient magnitudes $> 10$ on the logit scale paired with standard errors of similar size.
- A few fitted probabilities within $10^{-6}$ of 0 or 1.
- The Wald test reports $p \approx 1$ for a clearly important variable (Hauck–Donner).

The cure is **Firth's penalised likelihood** — it adds a Jeffreys prior to $\ell(\beta)$ and yields finite estimates under separation. It is available via the `firthlogist` package (`pip install firthlogist`). We touch on this in ex08.


---
# 7. Your turn <a id="7-your-turn"></a>

Solutions to all of the below are in the companion notebook `01ZLMA_ex07_LLM_solutions.ipynb`.

### Theory

**T1.** Show that the log-likelihood in (2.4.1) is **concave** in $\beta$. Hint: compute the Hessian and show it is $-X^\top W X$ with $W \succeq 0$.

**T2.** For the **probit** link, derive the score vector and the expected Fisher information. Verify that the canonical-link simplification no longer holds.

**T3.** Show that the variance function of the Binomial family is $V(\mu) = \mu(n - \mu)/n$, and derive the corresponding IRLS weights for an arbitrary link $g$.

**T4.** Consider two logistic models, one with the logit link and one with the probit link, fit to the same data. Relate the two parameter vectors $\beta^{\text{logit}}$ and $\beta^{\text{probit}}$ via the rough identity $\beta^{\text{logit}} \approx 1.6\,\beta^{\text{probit}}$. Derive this factor from the ratio of the standard deviations of the two latent distributions.

**T5.** A sample of $n = 100$ patients has been followed; the coefficient of `age` in a univariate logistic regression is $\hat\beta_{\text{age}} = 0.03$ with $\mathrm{se} = 0.01$. Compute the OR per 1-year and per 10-year increase, and the 95% Wald CI for both.

### Applied — Titanic

**A1.** Fit a model `survived ~ sex + pclass + embarked + age10` and report the odds ratios with 95% CI.
- Which 10-year age band has the greatest effect on survival odds? (Hint: fit an alternative model where you cut age into bands and compare.)

**A2.** Add a `sex:pclass` interaction. Write out the OR of *survival* for males relative to females, for each `pclass` separately.

**A3.** Perform an LRT comparing `survived ~ sex + pclass` against `survived ~ sex * pclass`. Does the interaction add significant explanatory power?

### Applied — Heart disease

**A4.** Starting from `m_logit` in Section 6, run a stepwise model reduction (by LRT). Which predictors can be dropped?

**A5.** Re-run Section 6.2 using `m_cloglog`. Do the binned residuals look better or worse?

**A6.** Compute the Hosmer–Lemeshow $\hat C$ statistic for the logit, probit, and cloglog fits. Which link gives the best fit?

### Monte Carlo — verifying the asymptotics

The theorems of Section 2 all have the form "as $n \to \infty$, such-and-such happens." Those claims are easy to check with simulation. In each of the tasks below, set up a data-generating process, simulate $K \approx 10^3$ replications, and report an empirical summary. Solutions with commentary are in the companion notebook.

**MC1 — Asymptotic normality of the MLE.** Fix $\beta = (-0.5, 1.0, -0.7)$ and two independent $\mathcal{N}(0, 1)$ predictors. For $n \in \{50, 200, 1000\}$, draw $K$ replications, fit the logistic regression, and plot a QQ-plot of the standardised estimator $(\hat\beta_1 - \beta_1)/\sqrt{(\mathcal{I}(\beta)^{-1})_{11}}$ against $\mathcal{N}(0,1)$. Does the fit improve with $n$?

**MC2 — Wald CI coverage.** For the same data-generating process and $n \in \{30, 100, 500\}$, compute the empirical coverage of the 95% Wald CI for $\beta_1$. Does coverage approach the nominal $0.95$ from below or above? At which $n$ is the deviation largest?

**MC3 — Size and power of the three tests.** Fix the design matrix structure but vary the true coefficient $\beta_2 \in \{0, 0.2, 0.4, 0.6\}$. For each of Wald / LRT / Rao and each $n \in \{50, 500\}$, record the empirical rejection rate at nominal level $0.05$. Under the null ($\beta_2 = 0$) this is the *size* of the test; under alternatives it is the *power*. Which test has size closest to nominal at small $n$? Which has the highest power?

**MC4 — Fisher information vs. empirical covariance.** Fit the model $K = 2000$ times at $n = 500$ and compute the empirical covariance matrix $\mathrm{Cov}(\hat\beta)$. Compare it entrywise against the theoretical $(X^\top W X)^{-1}$ evaluated at the true $\beta$ and averaged over the covariate distribution. Report the ratio.

**MC5 — Link-function misspecification.** Generate data from a **probit** model with $\beta^{\text{probit}} = (-0.3, 0.7)$. Fit both logit and probit and record the fitted coefficients. Verify numerically the scale factor $\beta^{\text{logit}} / \beta^{\text{probit}} \approx \pi / \sqrt{3}$ from T4, and compare the predicted probabilities (RMSE).

**MC6 — IRLS convergence and conditioning.** Generate correlated predictors $x_1, x_2$ with Pearson correlation $\rho \in \{0, 0.5, 0.9, 0.99, 0.999\}$. For each $\rho$, fit the logistic GLM many times and record (a) the number of IRLS iterations to convergence and (b) the condition number of $X^\top X$. Does IRLS slow down with collinearity? How severe is the effect?


---
# 8. Summary and transition to ex08 <a id="8-summary"></a>

This notebook covered:
* The full theoretical setup of the Bernoulli / Binomial GLM — exponential-family form, three link functions, likelihood, score, Fisher information, IRLS, asymptotics, deviance — and the three canonical tests (Wald, LRT, Rao). 
* Two real datasets illustrated the workflow: Titanic for discrete predictors and odds-ratio interpretation, Cleveland Heart for continuous predictors
* Diagnostics (Pearson / deviance / binned residuals, leverage, Cook's distance, Hosmer–Lemeshow).

**Exercise 08** will cover:

- ROC curves, AUC, calibration plots
- cut-off choice and the confusion matrix
- overdispersion and the quasi-binomial family
- Firth's penalised likelihood for (quasi-)separation
- marginal effects and average predicted probabilities
